# BÁO CÁO THỰC NGHIỆM
## Xấp xỉ hàm số bằng mạng nơ-ron MLP



## Tóm tắt

Báo cáo trình bày quá trình xây dựng và đánh giá mạng nơ-ron truyền thẳng nhiều tầng (Multilayer Perceptron – MLP) cho bài toán xấp xỉ một hàm mật độ xác suất Gaussian hai biến. Mô hình được triển khai bằng PyTorch, sử dụng hàm mất mát sai số bình phương trung bình (Mean Squared Error – MSE) và quy trình huấn luyện có tập validation để lựa chọn checkpoint phù hợp.

Các nội dung đánh giá tập trung vào chất lượng xấp xỉ của mô hình, độ ổn định giữa các lần chạy với seed khác nhau, thời gian huấn luyện và những hạn chế của phương pháp. Các chỉ số được trình bày dựa trên kết quả thực nghiệm lưu trong notebook.

**Từ khóa:** MLP, PyTorch, xấp xỉ hàm số, Gaussian hai biến, MSE, độ ổn định mô hình.

# 1. Đặt vấn đề và mục tiêu

## 1.1. Đặt vấn đề

Mạng nơ-ron truyền thẳng nhiều tầng (Multilayer Perceptron – MLP) là một mô hình học máy có khả năng học các quan hệ phi tuyến giữa đầu vào và đầu ra. Trong bài toán xấp xỉ hàm số, mạng được huấn luyện trên các cặp dữ liệu đầu vào và giá trị mục tiêu để dự đoán giá trị của hàm tại những điểm chưa được quan sát.

Bài tập này xem xét bài toán xấp xỉ hàm mật độ Gaussian hai biến. Đây là một hàm phi tuyến có giá trị thay đổi theo hai biến đầu vào. Mục tiêu là xây dựng mạng MLP có hai đầu vào và một đầu ra, sau đó đánh giá mức độ gần nhau giữa giá trị dự đoán của mạng và giá trị tính trực tiếp từ hàm Gaussian.

## 1.2. Mục tiêu

Các mục tiêu chính của thực nghiệm gồm:

- Xây dựng hàm sinh dữ liệu từ hàm Gaussian hai chiều.
- Cài đặt mạng MLP bằng PyTorch.
- Huấn luyện mạng bằng hàm mất mát MSE và thuật toán tối ưu phù hợp.
- Sử dụng tập validation để theo dõi quá trình huấn luyện và lựa chọn mô hình.
- Đánh giá mô hình bằng MSE, MAE và RMSE trên tập test.
- Khảo sát ảnh hưởng của quy mô dữ liệu, độ rộng tầng ẩn và độ sâu mạng.
- Đánh giá độ ổn định của kết quả khi thay đổi random seed.
- Phân tích hạn chế của mô hình và đề xuất hướng cải tiến.

## 1.3. Phạm vi thực nghiệm

Bài toán được thực hiện trong phạm vi xấp xỉ một hàm Gaussian hai biến đã biết công thức. Do đó, giá trị mục tiêu của mỗi mẫu có thể được tính trực tiếp từ hàm toán học. Sai số giữa giá trị thật và giá trị dự đoán là cơ sở để đánh giá chất lượng xấp xỉ của mạng.

## 2. Cơ sở lý thuyết và dữ liệu

### 2.1. Phát biểu bài toán

Xét hàm mật độ xác suất Gaussian hai chiều:

$$
f(\mathbf{x}) =
\frac{1}{2\pi\sqrt{\det(C)}}
\exp\left(
-\frac{1}{2}
(\mathbf{x}-\boldsymbol{\mu})^T
C^{-1}
(\mathbf{x}-\boldsymbol{\mu})
\right)
$$

Trong đó, vector đầu vào là:

$$
\mathbf{x} =
\begin{bmatrix}
x_1\\
x_2
\end{bmatrix}
$$

Vector trung bình được xác định bởi:

$$
\boldsymbol{\mu} =
\begin{bmatrix}
0\\
0
\end{bmatrix}
$$

Ma trận hiệp phương sai:

$$
C =
\begin{bmatrix}
0.3 & -0.5\\
-0.5 & 2
\end{bmatrix}
$$

Miền sinh dữ liệu được quy định như sau:

$$
-2 \leq x_1 \leq 2
$$

$$
-4 \leq x_2 \leq 4
$$

Với mỗi cặp đầu vào \((x_1,x_2)\), giá trị mục tiêu được tính bằng:

$$
y = f(x_1,x_2)
$$

Mục tiêu là huấn luyện MLP để tạo ra dự đoán \(\hat{y}\) gần với giá trị thực \(y\).

### 2.2. Ý nghĩa các tham số

- \(\boldsymbol{\mu}\): vector trung bình, xác định vị trí trung tâm của phân phối.
- \(C\): ma trận hiệp phương sai, mô tả phương sai của từng biến và mối quan hệ giữa hai biến.
- \(\det(C)\): định thức của ma trận hiệp phương sai.
- \(C^{-1}\): ma trận nghịch đảo của ma trận hiệp phương sai.
- \(\exp(\cdot)\): hàm số mũ tự nhiên.

Trong bài toán này, phần tử ngoài đường chéo của ma trận hiệp phương sai mang giá trị âm, thể hiện mối tương quan âm giữa hai biến theo cấu hình đã cho.


### 2.3. Sinh và chia dữ liệu

Dữ liệu được tạo bằng cách lấy các giá trị \(x_1\) và \(x_2\) trong miền xác định, sau đó tính nhãn \(y\) bằng hàm Gaussian hai chiều.

Mỗi mẫu dữ liệu có dạng:

$$
\mathbf{x}^{(i)} =
\begin{bmatrix}
x_1^{(i)}\\
x_2^{(i)}
\end{bmatrix},
\qquad
y^{(i)} = f(x_1^{(i)},x_2^{(i)})
$$

Dữ liệu được chia thành ba tập:

1. **Training set:** sử dụng để cập nhật trọng số và bias của mô hình.
2. **Validation set:** sử dụng để theo dõi khả năng tổng quát hóa và lựa chọn checkpoint.
3. **Test set:** sử dụng để đánh giá cuối cùng sau khi đã lựa chọn cấu hình.

Theo yêu cầu bài tập, tập kiểm tra gồm 100 mẫu độc lập. Các tập dữ liệu cần được sinh và quản lý sao cho không bị trùng lặp ngoài ý muốn.

Để bảo đảm khả năng tái lập, chương trình sử dụng random seed được xác định trước. Khi so sánh các cấu hình, những yếu tố không phải biến đang khảo sát cần được giữ nhất quán trong phạm vi hợp lý.

### 2.4. Trực quan hóa dữ liệu

Dữ liệu có thể được thể hiện bằng biểu đồ phân tán ba chiều, với hai trục đầu vào \(x_1\), \(x_2\) và trục đầu ra \(y\). Ngoài ra, bề mặt hàm Gaussian thực có thể được vẽ để quan sát hình dạng của hàm mục tiêu.

**Hình 1.** Biểu đồ phân bố dữ liệu Gaussian hai chiều.

**Hình 2.** Bề mặt ba chiều của hàm Gaussian thực.

*Chèn các biểu đồ được tạo từ notebook vào vị trí này.*

## 3. Mô hình và phương pháp huấn luyện

### 3.1. Kiến trúc MLP

MLP gồm một tầng đầu vào, một hoặc nhiều tầng ẩn và một tầng đầu ra. Với bài toán đang xét, tầng đầu vào nhận hai đặc trưng \(x_1\) và \(x_2\), trong khi tầng đầu ra tạo ra một giá trị dự đoán \(\hat{y}\).

Một tầng tuyến tính được biểu diễn bằng:

$$
\mathbf{z} = W\mathbf{x}+\mathbf{b}
$$

Trong đó, \(W\) là ma trận trọng số và \(\mathbf{b}\) là vector bias.

Sau phép biến đổi tuyến tính, hàm kích hoạt được áp dụng để mô hình học các quan hệ phi tuyến. Nếu sử dụng hàm tanh, đầu ra tầng ẩn là:

$$
\mathbf{h} = \tanh(W\mathbf{x}+\mathbf{b})
$$

Đối với mạng có nhiều tầng ẩn, đầu ra của tầng trước trở thành đầu vào của tầng tiếp theo. Tầng cuối cùng sử dụng phép biến đổi tuyến tính để tạo ra dự đoán.

### 3.2. Hàm mất mát MSE

Sai số bình phương trung bình (Mean Squared Error – MSE) được sử dụng làm hàm mất mát trong quá trình huấn luyện:

$$
MSE = \frac{1}{N}
\sum_{i=1}^{N}
(y_i-\hat{y}_i)^2
$$

Trong đó:

- \(N\) là số mẫu dữ liệu.
- \(y_i\) là giá trị mục tiêu thực.
- \(\hat{y}_i\) là giá trị dự đoán của mô hình.

MSE càng nhỏ thì sai số bình phương trung bình giữa dự đoán và giá trị mục tiêu càng thấp.

### 3.3. Quy trình huấn luyện

Quy trình huấn luyện được triển khai theo các bước:

1. Đưa batch dữ liệu vào mô hình để tính dự đoán.
2. Tính hàm mất mát giữa giá trị dự đoán và giá trị thực.
3. Xóa gradient của bước trước.
4. Lan truyền ngược để tính gradient.
5. Cập nhật tham số bằng bộ tối ưu.
6. Đánh giá trên tập validation và lưu checkpoint có validation loss tốt nhất.

Trong PyTorch, mô hình được định nghĩa bằng `torch.nn.Module`, các tầng tuyến tính sử dụng `nn.Linear`, và việc tối ưu được thực hiện bằng bộ tối ưu như Adam.

Tập validation được sử dụng để theo dõi quá trình học và hỗ trợ dừng sớm nếu được cấu hình. Tập test không được sử dụng trong quá trình cập nhật trọng số hoặc lựa chọn siêu tham số.

### 3.4. Các chỉ số đánh giá

Ngoài MSE, mô hình có thể được đánh giá bằng MAE và RMSE.

**Mean Absolute Error (MAE):**

$$
MAE = \frac{1}{N}
\sum_{i=1}^{N}|y_i-\hat{y}_i|
$$

MAE thể hiện độ lớn trung bình của sai số tuyệt đối.

**Root Mean Squared Error (RMSE):**

$$
RMSE =
\sqrt{
\frac{1}{N}
\sum_{i=1}^{N}(y_i-\hat{y}_i)^2
}
$$

RMSE là căn bậc hai của MSE và có cùng đơn vị với giá trị đầu ra.

Khi so sánh các mô hình, cần sử dụng cùng tập kiểm tra và cùng cách tính chỉ số. Nếu chạy nhiều random seed, kết quả nên được báo cáo bằng giá trị trung bình và độ lệch chuẩn.

Với \(n\) lần chạy, trung bình của một chỉ số \(m\) được tính như sau:

$$
\bar{m} = \frac{1}{n}\sum_{i=1}^{n}m_i
$$

Độ lệch chuẩn mẫu được tính bằng:

$$
s_m =
\sqrt{
\frac{1}{n-1}
\sum_{i=1}^{n}(m_i-\bar{m})^2
}
$$

## 6. Đánh giá tổng hợp

### 6.1. Các chỉ số tổng hợp

| Chỉ số | Giá trị trung bình | Độ lệch chuẩn |
|---|---:|---:|
| MSE | \(4.441111e-07) | \(1.912672e-07\) |
| MAE | \(5.228731e-04\) | \(1.090562e-04\) |
| RMSE | \(6.541711e-04\) | \(1.557468e-04) |
| Thời gian huấn luyện (giây) | 35.165 | 5.374382 |

**Bảng 5.** Các chỉ số tổng hợp được ghi nhận từ notebook.

### 6.2. Nhận xét kết quả

Các giá trị MSE, MAE và RMSE được ghi nhận đều có độ lớn nhỏ. Điều này cho thấy sai số dự đoán trung bình trong kết quả tổng hợp là thấp so với thang giá trị được báo cáo.

Tuy nhiên, để kết luận chính xác về chất lượng tổng quát hóa, cần xác nhận rằng những chỉ số này được tính trên tập test độc lập và không được sử dụng trong quá trình lựa chọn mô hình. Cần đồng thời đối chiếu kết quả với biểu đồ so sánh giá trị thực và giá trị dự đoán.

Độ lệch chuẩn phản ánh mức độ dao động của chỉ số qua các lần chạy được tổng hợp. Độ lệch chuẩn thấp hơn thường biểu thị kết quả ổn định hơn trong cùng điều kiện thực nghiệm.

## 7. Thảo luận

### 7.1. Ảnh hưởng của quy mô dữ liệu

Quy mô dữ liệu quyết định lượng thông tin mà mô hình có thể sử dụng để học hàm mục tiêu. Khi có nhiều mẫu hơn, mô hình có cơ hội quan sát hàm tại nhiều vị trí trong miền đầu vào. Tuy nhiên, mức cải thiện thực tế cần được xác nhận bằng kết quả thực nghiệm.

### 7.2. Ảnh hưởng của độ rộng

Số nơ-ron ở tầng ẩn ảnh hưởng đến khả năng biểu diễn của mạng. Mạng quá hẹp có thể chưa học được cấu trúc của hàm mục tiêu. Mạng rộng hơn có thể tăng khả năng biểu diễn, nhưng cũng làm tăng số lượng tham số và chi phí tính toán.

### 7.3. Ảnh hưởng của độ sâu

Số tầng ẩn quyết định mức độ phức tạp của chuỗi biến đổi phi tuyến. Mạng sâu có thể biểu diễn những quan hệ phức tạp, nhưng hiệu quả phụ thuộc vào cách thiết kế và tối ưu. Vì vậy, cần so sánh sai số và thời gian huấn luyện thay vì mặc định rằng mạng sâu hơn luôn tốt hơn.

### 7.4. Hạn chế

- Kết quả có thể phụ thuộc vào random seed và khởi tạo trọng số.
- Các cấu hình mạng có thể cần số epoch khác nhau để hội tụ.
- Số lượng mẫu test tương đối nhỏ nên kết quả đánh giá có thể có độ biến thiên.
- Kết luận về A, B và C cần dựa trên các bảng kết quả và biểu đồ thực tế.
- Nếu chỉ chạy một seed, chưa thể đánh giá đầy đủ độ ổn định giữa các lần chạy.

### 7.5. Hướng cải tiến

Có thể mở rộng nghiên cứu bằng cách tăng số lần chạy với các random seed khác nhau, khảo sát thêm hàm kích hoạt, điều chỉnh learning rate và so sánh nhiều bộ tối ưu. Ngoài ra, có thể trực quan hóa bề mặt dự đoán và bề mặt hàm thật trên cùng miền đầu vào để quan sát trực tiếp sai lệch của mô hình.

## 8. Kết luận

Bài tập đã đặt ra bài toán xấp xỉ hàm mật độ xác suất Gaussian hai chiều bằng mạng Perceptron đa tầng. Quy trình thực hiện bao gồm xây dựng dữ liệu, triển khai MLP bằng PyTorch, huấn luyện với hàm mất mát MSE và đánh giá bằng các chỉ số sai số.

Ba nhóm thực nghiệm A, B và C được thiết kế để khảo sát ảnh hưởng của số mẫu huấn luyện, số nơ-ron tầng ẩn và số tầng ẩn. Các kết quả cần được tổng hợp thành bảng và biểu đồ để so sánh công bằng giữa những cấu hình.

Để hoàn thiện kết luận, cần sử dụng kết quả thực nghiệm đã chạy để xác định cấu hình phù hợp nhất, đồng thời cân nhắc cả độ chính xác, tính ổn định và thời gian huấn luyện. Việc giữ riêng tập test và lưu lại cấu hình, log cùng checkpoint giúp tăng tính minh bạch và khả năng tái lập của nghiên cứu.